## Initialize all the necessary Libraries.

In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader, CSVLoader, JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter, TextSplitter, SentenceTransformersTokenTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain.chains import ConversationalRetrievalChain, RetrievalQA, RetrievalQAWithSourcesChain, LLMChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory, ConversationSummaryMemory
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

ModuleNotFoundError: No module named 'langchain.document_loaders'

## Initialize the LLM model.

In [7]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=os.getenv("GROK_API_KEY")
)

In [12]:
llm.invoke("Hello Buddy aaj apne kya khaya ??").content

'नमस्ते! मैं एक AI हूँ, इसलिए मैं खाने‑पीने की चीज़ें नहीं खाता। लेकिन अगर आप पूछ रहे हैं कि आज के लिए मैं क्या सुझा सकता हूँ, तो मैं कुछ स्वादिष्ट विकल्प बता सकता हूँ:\n\n- **दाल‑चावल** के साथ थोड़ा पापड़ और अचार – क्लासिक और हल्का।\n- **पनीर बटर मसाला** के साथ नान या रोटी – अगर आप कुछ क्रीमी और मसालेदार चाहते हैं।\n- **वेज़िटेबल स्टर‑फ्राई** और क्विनोआ – हेल्दी और जल्दी तैयार होने वाला।\n- **सूप** (टमाटर, मटर या मशरूम) के साथ टोस्ट – हल्का और पेट को सुकून देने वाला।\n\nआपने आज क्या खाया? कोई खास डिश या रेसिपी है जो आप शेयर करना चाहेंगे? 😄'

### Initialize the Memory

In [15]:
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True
)

## Load the Documents from the local Directory.

In [59]:
# from langchain.document_loaders import PyPDFLoader

# file_path = "data/heath_and_fitness.pdf"

# # 1️⃣ Create loader instance
# loader = PyPDFLoader(file_path)

# # 2 Iterate through all pages (Using the Lazy Load)
# # lazy_gen = loader.lazy_load()  # recreate generator if you want all pages
# # for doc in lazy_gen:
# #     print(doc.page_content)

# data = loader.load()

def load_documents(data_path: str):
    # Using the PyPDFLoader.
    # loader = PyPDFLoader(data_path+"heath_and_fitness.pdf")
    # data = loader.load()
    # return data

    # Using the Directory Loader (Benefit load all the pdf, json, csvs from a specific folder)
    loader = DirectoryLoader(
        data_path,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        # show_progress=True
    )
    return loader.load()

In [60]:
data[0]

Document(metadata={'producer': 'Acrobat Distiller 7.0.5 (Windows)', 'creator': 'PyPDF', 'creationdate': '2008-08-14T10:48:20-05:00', 'author': 'Black Dot Group', 'moddate': '2008-09-02T11:10:16-04:00', 'source': 'data/heath_and_fitness.pdf', 'total_pages': 10, 'page': 0, 'page_label': '76'}, page_content='Section\n2\nKey Points\n1 Components of Fitness\n2 Principles of Exercise\n3 Frequency, Intensity, Time, Type (FITT)\n4 Safety and Smart Training\n5 Nutrition and Diet\nHEALTH AND FITNESS\nTo every man there comes in his lifetime that special\nmoment when he is figuratively tapped on the shoulder\nand asked to do a very special thing—unique to him\nand his talents. What a tragedy if that moment finds him\nunprepared or unqualified for that work.\nSir Winston Churchill\ne\nPersonal Development Track\n8420010_PD2_p076-085  8/14/08  10:42 AM  Page 76')

In [61]:
print(len(data))

10


## Create the Chunks from the loaded data.

In [71]:
def data_splitting(data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30, separators=["\n", "\n\n", " ", ""])

    chunked_docs = text_splitter.split_documents(data)
    # print(len(chunked_docs))
    # print(chunked_docs)
    return chunked_docs

## Create the embedding for the data chunks.

In [78]:
def create_vector_store(chunks):
    # initialize the Embedding Model.
    embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")


    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )
    
    return vector_store

## Create the retriver 

In [82]:
def create_retriver(vector_store):
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k" : 6}
    )
    return retriever

# Initialize the Prompt Template

In [90]:
# Using the Simple Prompt Template.

# template = """
# You are an expert assistant. Answer the question based ONLY on the following context.
# If you don't know the answer, say "I don't have enough information."
# Context: {context}
# Question: {question}
# """

# prompt = PromptTemplate(template=template, input_variables=["context","questions"])

# formatted_prompt = prompt.format(
#     context="This is the relevant document from your vector store.",
#     question="What is the value of pi ??"
# )
# print(formatted_prompt)

# Using the ChatPromptTemplate
from langchain.prompts import ChatMessagePromptTemplate
from langchain.schema import SystemMessage, AIMessage, HumanMessage

prompt = ChatPromptTemplate([
    SystemMessage(content="""You are an expert assistant. Answer the question based ONLY on the following context.
                           If you don't know the answer, say "I don't have enough information.
                           Context: {context}
                           chat_history: {chat_history}"""),
    
    HumanMessage(content="{question}")
    ])

formatted_prompt = prompt.format_messages(
    context="This is the relevant document from your vector store.",
    chat_history="User: Hi\nAssistant: Hello!",
    question="which is the smallest thing in the world",
    
)

print(formatted_prompt)

[SystemMessage(content='You are an expert assistant. Answer the question based ONLY on the following context.\n                           If you don\'t know the answer, say "I don\'t have enough information.\n                           Context: {context}\n                           chat_history: {chat_history}', additional_kwargs={}, response_metadata={}), HumanMessage(content='{question}', additional_kwargs={}, response_metadata={})]


In [118]:
## Finally Create the LLM chain.
from langchain.chains import ConversationalRetrievalChain

def create_chain(retriever, llm ,memory):
    qa_rerank = ConversationalRetrievalChain.from_llm(llm,
                                                  retriever=retriever, 
                                                  memory=memory, 
                                                  # chain_type='map_rerank', 
                                                  verbose=True,
                                                # combine_docs_chain_kwargs={"prompt": formatted_prompt},
                                                 )
    return qa_rerank

## Full Pipeline Execution

In [119]:
def main():
    # Load the documents.
    docs = load_documents("data/")
    # print(data[6])

    chunks = data_splitting(docs)
    # print(chunks)

    vector_store = create_vector_store(chunks)
    # print(vector_store)

    retriever = create_retriver(vector_store)
    # print(retriever)

    qa_chain = create_chain(retriever, llm ,memory)
    # print(qa_chain)

    question = "what are the Components of Fitness ??"
    result_rerank1 = qa_chain({"question": question}) 
    print(f"Answer:\n {result_rerank1['answer']}")


if __name__ == "__main__":
    main()



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: What is a deep work
Assistant: **Deep work** is a term popularized by author and computer‑science professor **Cal Newport** in his 2016 book *Deep Work: Rules for Focused Success in a Distracted World*. It refers to:

| Aspect | Description |
|--------|-------------|
| **What it is** | A state of **distraction‑free, high‑concentration** activity that pushes your mental abilities to their limits. |
| **What it produces** | • **New value** (e.g., writing a research paper, solving a complex coding problem, designing a product). <br>• **Skill improvement** (the more you practice deep work, the better you become at thinking deeply). |
| **What it isn’t** | Routine, low‑cognitive‑load tasks such as checking email, scrolling social media, or other “shallow” work that 